# Initial refactored from R ../initial.py

In [4]:
import time
from datetime import datetime, timedelta, timezone
from typing import List, Tuple

import ccxt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

from dataloader import load_data


CSV_FILE = "advanced_orderflow_data.csv"
CSV_FILE = "/Users/spenser/Desktop/yihe/project/data/part2/advanced_orderflow_ws.csv"

def collect_orderflow_data(
    symbol: str = "BTC/USDT",
    poll_interval: int = 5,
    n_iterations: int = 15000,
    csv_file: str = CSV_FILE,
) -> None:
    """
    Collect order book snapshots from BinanceUS using ccxt and write
    microstructure features to a CSV file.

    Parameters
    ----------
    symbol : str
        Trading pair symbol, e.g. 'BTC/USDT'.
    poll_interval : int
        Seconds between API calls.
    n_iterations : int
        Number of snapshots to collect.
    csv_file : str
        Output CSV file path.
    """
    exchange = ccxt.binanceus()

    print(
        f"Starting advanced data collection for {symbol}. "
        f"Polling every {poll_interval}s for {n_iterations} iterations."
    )

    for i in range(1, n_iterations + 1):
        try:
            ob = exchange.fetch_order_book(symbol)

            bids_df = pd.DataFrame(ob["bids"], columns=["price", "amount"])
            asks_df = pd.DataFrame(ob["asks"], columns=["price", "amount"])

            if bids_df.empty or asks_df.empty:
                print(
                    f"[{datetime.now().strftime('%H:%M:%S')}] "
                    "Empty order book, skipping."
                )
                time.sleep(poll_interval)
                continue

            best_bid = bids_df["price"].iloc[0]
            best_ask = asks_df["price"].iloc[0]
            bid_vol_1 = bids_df["amount"].iloc[0]

            ask_vol_1 = asks_df["amount"].iloc[0]

            mid_price = (best_bid + best_ask) / 2
            spread = best_ask - best_bid
            fractional_price = mid_price - int(mid_price)

            micro_price = (best_bid * ask_vol_1 + best_ask * bid_vol_1) / (
                bid_vol_1 + ask_vol_1
            )

            obi_1 = (bid_vol_1 - ask_vol_1) / (bid_vol_1 + ask_vol_1)

            top_bids_10 = bids_df.head(10)
            top_asks_10 = asks_df.head(10)
            bid_vol_10 = top_bids_10["amount"].sum()
            ask_vol_10 = top_asks_10["amount"].sum()
            depth_10 = bid_vol_10 + ask_vol_10
            obi_10 = (bid_vol_10 - ask_vol_10) / (bid_vol_10 + ask_vol_10)

            obi_diff = obi_10 - obi_1
            depth_imbalance_10 = (bid_vol_10 - ask_vol_10) / depth_10 if depth_10 != 0 else 0.0

            current_time = datetime.now()

            row_data = pd.DataFrame(
                [
                    {
                        "timestamp": current_time,
                        "mid_price": mid_price,
                        "micro_price": micro_price,
                        "fractional_price": fractional_price,
                        "spread": spread,
                        "obi_1": obi_1,
                        "obi_10": obi_10,
                        "obi_diff": obi_diff,
                        "bid_vol_10": bid_vol_10,
                        "ask_vol_10": ask_vol_10,
                        "depth_10": depth_10,
                        "depth_imbalance_10": depth_imbalance_10,
                    }
                ]
            )

            header = not pd.io.common.file_exists(csv_file)
            row_data.to_csv(csv_file, mode="a", index=False, header=header)

            print(
                f"[{current_time.strftime('%H:%M:%S')}] "
                f"Iter {i}/{n_iterations} | "
                f"Mid: {mid_price:.2f} | OBI(10): {obi_10:+.2f} | Spread: {spread:.2f}"
            )

        except Exception as e:
            print(
                f"[{datetime.now().strftime('%H:%M:%S')}] "
                f"API error ({type(e).__name__}: {e}). Skipping iteration."
            )

        time.sleep(poll_interval)

    print("Data collection complete.")


def load_and_prepare_data(
    csv_file: str = CSV_FILE,
    horizon: int = 5,
) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, List[str]]:
    """
    Compatibility wrapper.

    The project now uses a single shared loader (`dataloader.py`) so that
    training/inference/visualization all see the same feature set and ordering.

    Note: `horizon` is ignored by the shared loader (it mirrors the R script’s
    forward-window logic).
    """
    return load_data(csv_file)


def train_test_split_time_series(
    X: np.ndarray, y: np.ndarray, train_ratio: float = 0.7
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Simple time-based train/test split.
    """
    n = len(y)
    split_idx = int(n * train_ratio)
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    return X_train, X_test, y_train, y_test


def run_logistic_regression(
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray,
) -> None:
    """
    Fit and evaluate an L2-regularized logistic regression model.
    """
    if X_train.size == 0 or X_test.size == 0:
        print("Skipping logistic regression: no samples available.")
        return
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
        n_jobs=-1,
    )

    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    print("=== Logistic Regression (L2) ===")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))
    print("\nClassification report:\n", classification_report(y_test, y_pred))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


def run_random_forest(
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray,
) -> None:
    """
    Fit and evaluate a Random Forest classifier.
    """
    if X_train.size == 0 or X_test.size == 0:
        print("Skipping random forest: no samples available.")
        return

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=42,
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print("=== Random Forest ===")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))
    print("\nClassification report:\n", classification_report(y_test, y_pred))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


def time_series_cross_validation(
    X_train: np.ndarray, y_train: np.ndarray, n_splits: int = 5
) -> None:
    """
    Time-series cross-validation for logistic regression as an example.
    """
    # If there is only one class in the training data, CV is not meaningful.
    unique_classes = np.unique(y_train)
    if unique_classes.size < 2:
        print(
            "Skipping time-series CV: training data contains only one class "
            f"{unique_classes}."
        )
        return

    tscv = TimeSeriesSplit(n_splits=n_splits)
    scaler = StandardScaler()
    cv_acc: List[float] = []

    print("=== Time Series Cross-Validation (Logistic Regression) ===")
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train), 1):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        # Some folds may still have a single class due to small sample size.
        if np.unique(y_tr).size < 2 or np.unique(y_val).size < 2:
            print(f"Fold {fold}: skipped (only one class in this fold).")
            continue

        X_tr_s = scaler.fit_transform(X_tr)
        X_val_s = scaler.transform(X_val)

        model = LogisticRegression(
            penalty="l2", C=1.0, solver="lbfgs", max_iter=1000, n_jobs=-1
        )
        model.fit(X_tr_s, y_tr)
        y_val_pred = model.predict(X_val_s)
        acc = accuracy_score(y_val, y_val_pred)
        cv_acc.append(acc)
        print(f"Fold {fold} accuracy: {acc:.3f}")

    print("Mean CV accuracy:", float(np.mean(cv_acc)))


def main() -> None:
    """
    Entry point: assumes CSV data has already been collected, then
    builds features, trains models, and reports performance.

    If you want to collect fresh data first, call collect_orderflow_data()
    before running the modeling steps.
    """
    # Uncomment this line if you want to (re)collect data from the exchange:
    # collect_orderflow_data(symbol="BTC/USDT", poll_interval=5, n_iterations=15000)

    df, X, y, feature_cols = load_and_prepare_data(CSV_FILE, horizon=5)
    print(f"Loaded {len(df)} rows with {len(feature_cols)} features.")

    if len(df) == 0 or X.size == 0:
        print(
            "No usable rows after feature/target construction. "
            "You may need to collect more data first."
        )
        return

    X_train, X_test, y_train, y_test = train_test_split_time_series(
        X, y, train_ratio=0.7
    )
    print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

    if X_train.size == 0 or X_test.size == 0:
        print(
            "Train/test split resulted in empty sets. "
            "Collect more data before modeling."
        )
        return

    time_series_cross_validation(X_train, y_train, n_splits=5)
    run_logistic_regression(X_train, X_test, y_train, y_test)
    run_random_forest(X_train, X_test, y_train, y_test)


if __name__ == "__main__":
    main()

# 

Loaded 407549 rows with 11 features.
Train size: (285284, 11), Test size: (122265, 11)
=== Time Series Cross-Validation (Logistic Regression) ===
Fold 1 accuracy: 0.580
Fold 2 accuracy: 0.628
Fold 3 accuracy: 0.660
Fold 4 accuracy: 0.625
Fold 5 accuracy: 0.586
Mean CV accuracy: 0.6160809304477675
=== Logistic Regression (L2) ===
Accuracy: 0.561190855927698
ROC-AUC: 0.5728984233755645

Classification report:
               precision    recall  f1-score   support

           0       0.56      0.87      0.68     66141
           1       0.56      0.19      0.29     56124

    accuracy                           0.56    122265
   macro avg       0.56      0.53      0.49    122265
weighted avg       0.56      0.56      0.50    122265

Confusion matrix:
 [[57791  8350]
 [45301 10823]]
=== Random Forest ===
Accuracy: 0.5440723019670388
ROC-AUC: 0.5437364728970032

Classification report:
               precision    recall  f1-score   support

           0       0.56      0.78      0.65     6614

# Dataloader ../dataloader.py

In [5]:
from __future__ import annotations
from dataclasses import dataclass
from typing import List, Tuple
import numpy as np
import pandas as pd


@dataclass(frozen=True)
class LoadedData:
    df: pd.DataFrame
    X: np.ndarray
    y: np.ndarray
    feature_cols: List[str]


def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map: dict[str, str] = {}
    if "Mid Price" in df.columns:
        rename_map["Mid Price"] = "mid_price"
    if "Micro Price" in df.columns:
        rename_map["Micro Price"] = "micro_price"
    if "fractional price" in df.columns:
        rename_map["fractional price"] = "fractional_price"
    if rename_map:
        df = df.rename(columns=rename_map)
    return df


def load_clean_r_style(csv_file: str) -> LoadedData:
    """
    Mirror the R cleaning logic (see `transformer.py` history / `stuff/clean.R`).

    Returns a model-ready dataframe, numeric feature matrix X, binary label y,
    and the feature column names in the exact order used to build X.
    """
    df = pd.read_csv(csv_file)
    df = _normalize_columns(df)

    if "Time" not in df.columns:
        raise ValueError("Expected a 'Time' column in the CSV.")

    # Parse Time: supports both MM:SS.s (e.g. "36:50.6") and HH:MM:SS (e.g. "03:49:45").
    time_str = df["Time"].astype(str)
    time_split = time_str.str.split(":", expand=True)
    is_hms = time_str.str.count(":") == 2  # HH:MM:SS has 2 colons
    mmss_secs = (
        pd.to_numeric(time_split[0], errors="coerce").fillna(0) * 60
        + pd.to_numeric(time_split[1], errors="coerce").fillna(0)
    )
    if time_split.shape[1] >= 3:
        hms_secs = (
            pd.to_numeric(time_split[0], errors="coerce").fillna(0) * 3600
            + pd.to_numeric(time_split[1], errors="coerce").fillna(0) * 60
            + pd.to_numeric(time_split[2], errors="coerce").fillna(0)
        )
        seconds_in_hour = np.where(is_hms, hms_secs, mmss_secs)
    else:
        seconds_in_hour = mmss_secs.values
    time_step = pd.Series(seconds_in_hour).diff()
    time_step = np.where((~np.isnan(time_step)) & (time_step < 0), time_step + 3600, time_step)
    elapsed_seconds = pd.Series(time_step).fillna(0).cumsum()
    df["Elapsed_Seconds"] = elapsed_seconds

    # Backward / forward windows (match the R script’s indexing).
    df["time_passed_backward"] = df["Elapsed_Seconds"] - df["Elapsed_Seconds"].shift(24)
    df["time_passed_forward"] = df["Elapsed_Seconds"].shift(-6) - df["Elapsed_Seconds"]

    # Volatility over last 24 mid_price values if within ~120s.
    roll_std = df["mid_price"].rolling(window=24, min_periods=24).std()
    df["volatility_120s"] = np.where(df["time_passed_backward"] < 135, roll_std, np.nan)

    # OBI momentum: obi_10 - lag 24 if within ~120s.
    obi_10_lag = df["obi_10"].shift(24)
    df["obi_momentum"] = np.where(df["time_passed_backward"] < 135, df["obi_10"] - obi_10_lag, np.nan)

    # Future price / return and target class.
    future_price = np.where(df["time_passed_forward"] < 35, df["mid_price"].shift(-6), np.nan)
    df["future_price"] = future_price
    df["future_return"] = (df["future_price"] - df["mid_price"]) / df["mid_price"]
    df["target_class"] = np.where(df["future_return"] > 0, 1, -1)

    # Drop rows with any NA in model-critical columns.
    df_model_ready = df.dropna(
        subset=[
            "volatility_120s",
            "obi_momentum",
            "future_price",
            "future_return",
            "target_class",
        ]
    ).reset_index(drop=True)

    y = (df_model_ready["target_class"] == 1).astype(int).to_numpy()

    exclude_cols = {
        "future_price",
        "future_return",
        "target_class",
        "time_passed_forward",  # data leak: uses shift(-6), i.e. future elapsed time
    }
    feature_cols = [
        c
        for c in df_model_ready.columns
        if c not in exclude_cols and df_model_ready[c].dtype != "O"
    ]
    X = df_model_ready[feature_cols].to_numpy(dtype=float)

    return LoadedData(df=df_model_ready, X=X, y=y, feature_cols=feature_cols)


def load_data(csv_file: str) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, List[str]]:
    """
    Backward-compatible tuple return for existing callers.
    """
    out = load_clean_r_style(csv_file)
    return out.df, out.X, out.y, out.feature_cols



# Transfromer Training ../transformer.py

In [6]:
import argparse
import json
import os
import pickle
import random
import time
from pathlib import Path
from typing import Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

from dataloader import load_clean_r_style


# ---------------------------------------------------------------------------
# Output versioning
# ---------------------------------------------------------------------------

def get_next_version_dir(base: str = "outputs") -> Path:
    """Auto-increments outputs/v1, v2, v3 ... on each run."""
    base_path = Path(base)
    base_path.mkdir(exist_ok=True)
    existing = [d for d in base_path.iterdir() if d.is_dir() and d.name.startswith("v")]
    versions = []
    for d in existing:
        try:
            versions.append(int(d.name[1:]))
        except ValueError:
            pass
    next_v = max(versions, default=0) + 1
    out_dir = base_path / f"v{next_v}"
    out_dir.mkdir()
    print(f"Output directory: {out_dir}")
    return out_dir


# ---------------------------------------------------------------------------
# Positional Encoding
# ---------------------------------------------------------------------------

class PositionalEncoding(nn.Module):
    """Fixed sinusoidal positional encoding (Vaswani et al.)."""

    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-np.log(500.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)


class LearnablePositionalEncoding(nn.Module):
    """Learnable positional embeddings (trained with the model)."""

    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1) -> None:
        super().__init__()
        self.pe = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------

class TimeSeriesTransformer(nn.Module):
    def __init__(
        self,
        input_dim: int,
        d_model: int = 64,
        nhead: int = 4,
        num_layers: int = 2,
        dim_feedforward: int = 256,
        dropout: float = 0.2,
        pos_encoding: str = "sinusoidal",
        attn_diagonal_bias: float = 0.0,
    ) -> None:
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        if pos_encoding == "sinusoidal":
            self.pos_enc = PositionalEncoding(d_model, dropout=dropout)
        elif pos_encoding == "positional":
            self.pos_enc = LearnablePositionalEncoding(d_model, dropout=dropout)
        else:
            raise ValueError(f"pos_encoding must be 'sinusoidal' or 'positional', got {pos_encoding!r}")
        self.encoder = TransformerEncoderWithAttn(
            d_model=d_model,
            nhead=nhead,
            num_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            attn_diagonal_bias=attn_diagonal_bias,
        )
        self.cls_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1),
        )

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        h = self.input_proj(x)
        h = self.pos_enc(h)
        if return_attn:
            h, attn = self.encoder(h, return_attn=True)
        else:
            h = self.encoder(h)
            attn = None
        last = h[:, -1, :]
        logits = self.cls_head(last).squeeze(-1)
        if return_attn:
            return logits, attn
        return logits


class TransformerEncoderLayerWithAttn(nn.Module):
    def __init__(
        self,
        d_model: int,
        nhead: int,
        dim_feedforward: int,
        dropout: float,
        attn_diagonal_bias: float = 0.0,
    ) -> None:
        super().__init__()
        self.attn_diagonal_bias = attn_diagonal_bias
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True,
        )
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )

    def _get_attn_mask(self, T: int, device: torch.device) -> Optional[torch.Tensor]:
        """Bias favoring diagonal: -lambda * |i-j|. Encourages local attention."""
        if self.attn_diagonal_bias <= 0:
            return None
        i = torch.arange(T, device=device, dtype=torch.float32)
        j = torch.arange(T, device=device, dtype=torch.float32)
        dist = (i.unsqueeze(1) - j.unsqueeze(0)).abs()
        return (-self.attn_diagonal_bias * dist).to(device)

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        x_norm = self.norm1(x)
        T = x.size(1)
        attn_mask = self._get_attn_mask(T, x.device)
        attn_out, attn_w = self.self_attn(
            x_norm,
            x_norm,
            x_norm,
            attn_mask=attn_mask,
            need_weights=return_attn,
            average_attn_weights=False,
        )
        x = x + self.dropout1(attn_out)
        x = x + self.dropout2(self.ff(self.norm2(x)))
        if return_attn:
            return x, attn_w  # (B, H, T, T)
        return x


class TransformerEncoderWithAttn(nn.Module):
    def __init__(
        self,
        d_model: int,
        nhead: int,
        num_layers: int,
        dim_feedforward: int,
        dropout: float,
        attn_diagonal_bias: float = 0.0,
    ) -> None:
        super().__init__()
        self.layers = nn.ModuleList(
            [
                TransformerEncoderLayerWithAttn(
                    d_model=d_model,
                    nhead=nhead,
                    dim_feedforward=dim_feedforward,
                    dropout=dropout,
                    attn_diagonal_bias=attn_diagonal_bias,
                )
                for _ in range(num_layers)
            ]
        )
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        last_attn = None
        for layer in self.layers:
            if return_attn:
                x, last_attn = layer(x, return_attn=True)
            else:
                x = layer(x, return_attn=False)
        x = self.final_norm(x)
        if return_attn:
            return x, last_attn
        return x


def save_attention_heatmap(
    attn: torch.Tensor,
    out_path: Path,
    sample_idx: int = 0,
    head: int = 0,
    title: str = "Self-attention heatmap (last layer)",
) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    attn_np = attn.detach().cpu().numpy()
    if attn_np.ndim != 4:
        raise ValueError(f"Expected attn with shape (B,H,T,T), got {attn_np.shape}")
    A = attn_np[sample_idx, head]
    plt.figure(figsize=(7, 6))
    sns.heatmap(A, cmap="viridis")
    plt.title(title)
    plt.xlabel("Key timestep")
    plt.ylabel("Query timestep")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


# ---------------------------------------------------------------------------
# Dataset helpers
# ---------------------------------------------------------------------------

def make_sequence_dataset(
    X: np.ndarray,
    y: np.ndarray,
    seq_len: int,
) -> Tuple[np.ndarray, np.ndarray]:
    if len(X) <= seq_len:
        raise ValueError("Not enough samples to create sequences.")
    X_seq, y_seq = [], []
    for t in range(seq_len - 1, len(X)):
        X_seq.append(X[t - seq_len + 1 : t + 1])
        y_seq.append(y[t])
    return np.stack(X_seq), np.array(y_seq)


def compute_pos_weight(y: np.ndarray, device: torch.device) -> torch.Tensor:
    n_pos = y.sum()
    n_neg = len(y) - n_pos
    ratio = n_neg / max(n_pos, 1)
    print(f"  Class balance — pos: {int(n_pos)}, neg: {int(n_neg)}, pos_weight: {ratio:.3f}")
    return torch.tensor([ratio], dtype=torch.float, device=device)


def find_best_threshold(logits: np.ndarray, targets: np.ndarray) -> float:
    """Sweep [0.3, 0.7] and pick threshold with best macro F1."""
    probs = 1 / (1 + np.exp(-logits))
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.3, 0.71, 0.02):
        preds = (probs >= t).astype(int)
        f1 = f1_score(targets, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t)


def eval_auc(model, loader, device) -> float:
    """Return ROC-AUC of model on a DataLoader. Returns 0.5 if undefined."""
    model.eval()
    logits_list, targets_list = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits_list.append(model(xb.to(device)).cpu().numpy())
            targets_list.append(yb.numpy())
    logits  = np.concatenate(logits_list)
    targets = np.concatenate(targets_list)
    probs   = 1 / (1 + np.exp(-logits))
    try:
        return roc_auc_score(targets, probs)
    except ValueError:
        return 0.5


# ---------------------------------------------------------------------------
# Training curve plot
# ---------------------------------------------------------------------------

def save_training_curves(history: dict, out_dir: Path) -> None:
    """Save loss and AUC curves side-by-side as a PNG."""
    epochs = range(1, len(history["train_loss"]) + 1)
    best_ep = history.get("best_epoch_auc", None)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("Training History", fontsize=14, fontweight="bold")

    # Loss
    ax = axes[0]
    ax.plot(epochs, history["train_loss"], label="Train Loss", linewidth=2)
    ax.plot(epochs, history["val_loss"],   label="Val Loss",   linewidth=2, linestyle="--")
    if best_ep:
        ax.axvline(best_ep, color="red", linestyle=":", alpha=0.6, label=f"Best epoch ({best_ep})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("BCE Loss")
    ax.set_title("Loss over Epochs")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # AUC
    ax = axes[1]
    ax.plot(epochs, history["train_auc"], label="Train AUC", linewidth=2)
    ax.plot(epochs, history["val_auc"],   label="Val AUC",   linewidth=2, linestyle="--")
    ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5, label="Random baseline")
    if best_ep:
        ax.axvline(best_ep, color="red", linestyle=":", alpha=0.6, label=f"Best epoch ({best_ep})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("ROC-AUC")
    ax.set_title("AUC over Epochs")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    png_path = out_dir / "training_curves.png"
    plt.savefig(png_path, format="png", bbox_inches="tight", dpi=150)
    plt.close()
    print(f"  Saved training curves → {png_path}")


# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------

def train_transformer(
    csv_file: str = "advanced_orderflow_data.csv",
    horizon: int = 5,
    seq_len: int = 40,
    n_epochs: int = 50,
    batch_size: int = 64,
    lr: float = 1e-4,
    patience: int = 8,
    val_ratio: float = 0.15,
    outputs_base: str = "outputs",
    seed: Optional[int] = None,
    encoding: str = "sinusoidal",
    attn_diagonal_bias: float = 0.0,
) -> None:

    if seed is None:
        seed = int(time.time() * 1e6) % (2**31)
        print(f"Using random seed: {seed}")
    else:
        print(f"Using fixed seed: {seed}")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    out_dir         = get_next_version_dir(outputs_base)
    checkpoint_path = out_dir / "transformer_best.pt"
    scaler_path     = out_dir / "scaler.pkl"

    # 1. Load and clean data (mirror R clean.R logic)
    loaded = load_clean_r_style(csv_file)
    df, X, y, feature_cols = loaded.df, loaded.X, loaded.y, loaded.feature_cols
    print(f"Loaded {len(df)} rows with {len(feature_cols)} features.")
    if len(df) == 0 or X.size == 0:
        print("No usable rows.")
        return

    # 2. Temporal split
    n       = len(X)
    n_test  = int(n * 0.15)
    n_val   = int(n * val_ratio)
    n_train = n - n_val - n_test

    X_train_raw = X[:n_train];           y_train_raw = y[:n_train]
    X_val_raw   = X[n_train:n_train+n_val]; y_val_raw = y[n_train:n_train+n_val]
    X_test_raw  = X[n_train+n_val:];     y_test_raw  = y[n_train+n_val:]

    print(f"Split — train: {len(X_train_raw)}, val: {len(X_val_raw)}, test: {len(X_test_raw)}")

    # 3. Scale (fit only on train)
    scaler    = StandardScaler()
    X_train_s = scaler.fit_transform(X_train_raw)
    X_val_s   = scaler.transform(X_val_raw)
    X_test_s  = scaler.transform(X_test_raw)
    with open(scaler_path, "wb") as f:
        pickle.dump(scaler, f)
    print(f"  Saved scaler → {scaler_path}")

    # 4. Sequences
    try:
        X_train_seq, y_train_seq = make_sequence_dataset(X_train_s, y_train_raw, seq_len)
        X_val_seq,   y_val_seq   = make_sequence_dataset(X_val_s,   y_val_raw,   seq_len)
        X_test_seq,  y_test_seq  = make_sequence_dataset(X_test_s,  y_test_raw,  seq_len)
    except ValueError as e:
        print(f"Skipping: {e}"); return

    print(f"Sequences — train: {X_train_seq.shape}, val: {X_val_seq.shape}, test: {X_test_seq.shape}")
    print(f"Positional encoding: {encoding}")
    if attn_diagonal_bias > 0:
        print(f"Attention diagonal bias: {attn_diagonal_bias} (encourages local/diagonal attention)")

    # 5. DataLoaders
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    def to_loader(Xs, ys):
        ds = TensorDataset(torch.from_numpy(Xs).float(), torch.from_numpy(ys).float())
        return DataLoader(ds, batch_size=batch_size, shuffle=False)

    train_loader = to_loader(X_train_seq, y_train_seq)
    val_loader   = to_loader(X_val_seq,   y_val_seq)
    test_loader  = to_loader(X_test_seq,  y_test_seq)

    # 6. Model / loss / optimizer
    model = TimeSeriesTransformer(
        input_dim=X_train_s.shape[1],
        pos_encoding=encoding,
        attn_diagonal_bias=attn_diagonal_bias,
    ).to(device)
    pos_weight = compute_pos_weight(y_train_seq, device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3   # mode=max: we track AUC
    )

    # 7. Training — checkpoint by val AUC
    print("\n=== Transformer — Bitcoin Up/Down ===")
    history = {
        "train_loss": [], "val_loss": [],
        "train_auc":  [], "val_auc":  [],
        "best_epoch_auc": 1,
    }
    best_val_auc      = 0.0
    epochs_no_improve = 0

    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item() * xb.size(0)
        epoch_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_loss += criterion(model(xb), yb).item() * xb.size(0)
        val_loss /= len(val_loader.dataset)

        train_auc = eval_auc(model, train_loader, device)
        val_auc   = eval_auc(model, val_loader,   device)

        scheduler.step(val_auc)
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(epoch_loss)
        history["val_loss"].append(val_loss)
        history["train_auc"].append(train_auc)
        history["val_auc"].append(val_auc)

        print(
            f"Epoch {epoch:3d}/{n_epochs} | "
            f"loss {epoch_loss:.4f}/{val_loss:.4f} | "
            f"AUC {train_auc:.4f}/{val_auc:.4f} | "
            f"lr {current_lr:.2e}"
        )

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            epochs_no_improve = 0
            history["best_epoch_auc"] = epoch
            torch.save(model.state_dict(), checkpoint_path)
            print(f"  ✓ Checkpoint saved (val AUC={val_auc:.4f})")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    # 8. Save training curves
    save_training_curves(history, out_dir)

    # 9. Final test evaluation
    try:
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        print(f"\nLoaded best checkpoint (epoch {history['best_epoch_auc']}).")
    except Exception as e:
        print(f"Could not load checkpoint: {e}")

    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            all_logits.append(model(xb.to(device)).cpu().numpy())
            all_targets.append(yb.numpy())

    y_logits  = np.concatenate(all_logits)
    y_targets = np.concatenate(all_targets)

    val_logits_list, val_targets_list = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            val_logits_list.append(model(xb.to(device)).cpu().numpy())
            val_targets_list.append(yb.numpy())

    best_threshold = find_best_threshold(
        np.concatenate(val_logits_list),
        np.concatenate(val_targets_list),
    )
    print(f"\nBest threshold (val): {best_threshold:.2f}")

    y_prob = 1 / (1 + np.exp(-y_logits))
    y_pred = (y_prob >= best_threshold).astype(int)

    acc = accuracy_score(y_targets, y_pred)
    try:
        auc = roc_auc_score(y_targets, y_prob)
    except ValueError:
        auc = float("nan")

    print(f"\nTest Accuracy : {acc:.4f}")
    print(f"Test ROC-AUC  : {auc:.4f}")
    print("\nClassification Report:\n", classification_report(y_targets, y_pred, zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(y_targets, y_pred))

    # Save metrics summary
    summary_path = out_dir / "metrics_summary.txt"
    with open(summary_path, "w") as f:
        f.write(f"Version dir      : {out_dir}\n")
        f.write(f"CSV              : {csv_file}\n")
        f.write(f"Horizon          : {horizon}\n")
        f.write(f"Seq len          : {seq_len}\n")
        f.write(f"Pos encoding     : {encoding}\n")
        f.write(f"Attn diag bias   : {attn_diagonal_bias}\n")
        f.write(f"Seed             : {seed}\n")
        f.write(f"Best epoch (AUC) : {history['best_epoch_auc']}\n")
        f.write(f"Best val AUC     : {best_val_auc:.4f}\n")
        f.write(f"Threshold        : {best_threshold:.2f}\n")
        f.write(f"Test Accuracy    : {acc:.4f}\n")
        f.write(f"Test AUC         : {auc:.4f}\n\n")
        f.write(classification_report(y_targets, y_pred, zero_division=0))
    print(f"  Saved metrics summary → {summary_path}")

    # Save config for inference (live_inference, tsne can read attn_diagonal_bias)
    config_path = out_dir / "config.json"
    with open(config_path, "w") as f:
        json.dump(
            {
                "seq_len": seq_len,
                "encoding": encoding,
                "attn_diagonal_bias": attn_diagonal_bias,
                "seed": seed,
            },
            f,
            indent=2,
        )
    print(f"  Saved config → {config_path}")
    print(f"\nAll outputs saved to: {out_dir}/")

    # 10. Save a self-attention heatmap from the best model
    try:
        xb0, yb0 = next(iter(val_loader))
        xb0 = xb0.to(device)
        _, attn = model(xb0, return_attn=True)
        if attn is not None:
            heatmap_path = out_dir / "attention_heatmap.png"
            npy_path = out_dir / "attention_weights.npy"
            save_attention_heatmap(attn, heatmap_path, sample_idx=0, head=0)
            np.save(npy_path, attn.detach().cpu().numpy())
            print(f"  Saved attention heatmap → {heatmap_path}")
            print(f"  Saved attention weights → {npy_path}")
    except Exception as e:
        print(f"  Could not save attention heatmap: {e}")


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Bitcoin Up/Down Transformer")
    parser.add_argument("--csv",          type=str,   default="advanced_orderflow_data.csv")
    parser.add_argument("--horizon",      type=int,   default=5)
    parser.add_argument("--seq-len",      type=int,   default=40)
    parser.add_argument("--epochs",       type=int,   default=50)
    parser.add_argument("--batch-size",   type=int,   default=64)
    parser.add_argument("--lr",           type=float, default=1e-4)
    parser.add_argument("--patience",     type=int,   default=8)
    parser.add_argument("--val-ratio",    type=float, default=0.15)
    parser.add_argument("--outputs-base", type=str,   default="outputs")
    parser.add_argument(
        "--seed",
        type=int,
        default=None,
        metavar="N",
        help="Random seed for reproducibility. Omit for a random seed each run.",
    )
    parser.add_argument(
        "--encoding",
        type=str,
        default="sinusoidal",
        choices=["sinusoidal", "positional"],
        help="Positional encoding: 'sinusoidal' (fixed) or 'positional' (learnable)",
    )
    parser.add_argument(
        "--attn-diagonal-bias",
        type=float,
        default=0.0,
        metavar="LAMBDA",
        help="Attention bias favoring diagonal: -lambda*|i-j|. Use 0.1-0.5 to reduce collapse. 0=off.",
    )
    return parser.parse_args()


if __name__ == "__main__":
    args = parse_args()
    train_transformer(
        csv_file=args.csv,
        horizon=args.horizon,
        seq_len=args.seq_len,
        n_epochs=args.epochs,
        batch_size=args.batch_size,
        lr=args.lr,
        patience=args.patience,
        val_ratio=args.val_ratio,
        outputs_base=args.outputs_base,
        seed=args.seed,
        encoding=args.encoding,
        attn_diagonal_bias=args.attn_diagonal_bias,
    )


usage: ipykernel_launcher.py [-h] [--csv CSV] [--horizon HORIZON]
                             [--seq-len SEQ_LEN] [--epochs EPOCHS]
                             [--batch-size BATCH_SIZE] [--lr LR]
                             [--patience PATIENCE] [--val-ratio VAL_RATIO]
                             [--outputs-base OUTPUTS_BASE] [--seed N]
                             [--encoding {sinusoidal,positional}]
                             [--attn-diagonal-bias LAMBDA]
ipykernel_launcher.py: error: unrecognized arguments: --f=/Users/spenser/Library/Jupyter/runtime/kernel-v32e546fff6bfa7ddbfd62db19208659543e0addea.json


SystemExit: 2

/opt/anaconda3/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# TSNE ../tse.py

In [ ]:
import argparse
import pickle
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.manifold import TSNE

from transformer import TimeSeriesTransformer, make_sequence_dataset
from initial import load_and_prepare_data


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def find_latest_version(base: str = "outputs") -> Path:
    base_path = Path(base)
    dirs = [d for d in base_path.iterdir() if d.is_dir() and d.name.startswith("v")]
    if not dirs:
        raise FileNotFoundError(f"No versioned run folders found in '{base}'.")
    versions = []
    for d in dirs:
        try:
            versions.append((int(d.name[1:]), d))
        except ValueError:
            pass
    versions.sort(key=lambda x: x[0])
    return versions[-1][1]


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def plot_tsne(
    outputs_base: str = "outputs",
    version: int = None,
    csv_file: str = "advanced_orderflow_data.csv",
    seq_len: int = 40,
    horizon: int = 5,
    n_samples: int = 1500,
    perplexity: int = 30,
) -> None:

    # Locate version folder
    if version is not None:
        run_dir = Path(outputs_base) / f"v{version}"
    else:
        run_dir = find_latest_version(outputs_base)

    print(f"Loading from: {run_dir}")

    checkpoint_path = run_dir / "transformer_best.pt"
    scaler_path     = run_dir / "scaler.pkl"

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"No checkpoint at {checkpoint_path}")

    # Load scaler if available (use it the same way training did)
    scaler = None
    if scaler_path.exists():
        with open(scaler_path, "rb") as f:
            scaler = pickle.load(f)
        print("  Loaded scaler from run folder.")
    else:
        print("  Warning: no scaler.pkl found — using raw (unscaled) features.")

    # Load data
    df, X, y, feat_cols = load_and_prepare_data(csv_file=csv_file, horizon=horizon)
    print(f"Loaded {len(df)} rows, {len(feat_cols)} features.")

    # Apply same scaler used during training
    if scaler is not None:
        X = scaler.transform(X)

    # Build sequences
    X_seq, y_seq = make_sequence_dataset(X, y, seq_len)

    # Subsample (tSNE is O(n²))
    if len(X_seq) > n_samples:
        idx = np.random.default_rng(42).choice(len(X_seq), n_samples, replace=False)
        idx.sort()
        X_seq = X_seq[idx]
        y_seq = y_seq[idx]

    print(f"Running tSNE on {len(X_seq)} samples …")

    X_torch = torch.tensor(X_seq, dtype=torch.float)

    # Load model (read config for encoding and attn_diagonal_bias)
    import json
    d_in = X_seq.shape[-1]
    config = {}
    config_path = run_dir / "config.json"
    if config_path.exists():
        with open(config_path) as f:
            config = json.load(f)
    model = TimeSeriesTransformer(
        input_dim=d_in,
        pos_encoding=config.get("encoding", "sinusoidal"),
        attn_diagonal_bias=config.get("attn_diagonal_bias", 0.0),
    )
    model.load_state_dict(torch.load(checkpoint_path, map_location="cpu"))
    model.eval()

    # Extract representations from encoder
    with torch.no_grad():
        h = model.input_proj(X_torch)
        h = model.pos_enc(h)
        h = model.encoder(h)
        feats = h[:, -1, :]           # (N, d_model) — last timestep

    feats_np = feats.cpu().numpy()

    # tSNE
    tsne   = TSNE(n_components=2, perplexity=perplexity, random_state=42, n_iter=1000)
    X_2d   = tsne.fit_transform(feats_np)

    # -----------------------------------------------------------------------
    # Plot
    # -----------------------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(
        f"t-SNE of Transformer Encoder Representations  ({run_dir.name})",
        fontsize=13, fontweight="bold",
    )

    # Left: colour by true label
    ax = axes[0]
    sc = ax.scatter(
        X_2d[:, 0], X_2d[:, 1],
        c=y_seq, cmap="coolwarm", alpha=0.65, s=18, linewidths=0,
    )
    plt.colorbar(sc, ax=ax, label="True label (0=down, 1=up)")
    ax.set_title("Coloured by True Label")
    ax.set_xlabel("tSNE-1");  ax.set_ylabel("tSNE-2")
    ax.grid(True, alpha=0.2)

    # Right: colour by model confidence (sigmoid of logit)
    with torch.no_grad():
        logits = model.cls_head(feats).squeeze(-1).cpu().numpy()
    probs = 1 / (1 + np.exp(-logits))

    ax = axes[1]
    sc2 = ax.scatter(
        X_2d[:, 0], X_2d[:, 1],
        c=probs, cmap="RdYlGn", alpha=0.65, s=18, linewidths=0,
        vmin=0, vmax=1,
    )
    plt.colorbar(sc2, ax=ax, label="Model confidence (P(up))")
    ax.set_title("Coloured by Model Confidence")
    ax.set_xlabel("tSNE-1");  ax.set_ylabel("tSNE-2")
    ax.grid(True, alpha=0.2)

    plt.tight_layout()

    png_path = run_dir / "tsne_plot.png"
    plt.savefig(png_path, format="png", bbox_inches="tight", dpi=150)
    plt.close()
    print(f"  Saved tSNE plot → {png_path}")


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="tSNE of transformer encoder reps.")
    parser.add_argument("--outputs-base", type=str,  default="outputs")
    parser.add_argument("--version",      type=int,  default=None,   help="Run version (default: latest)")
    parser.add_argument("--csv",          type=str,  default="advanced_orderflow_data.csv")
    parser.add_argument("--seq-len",      type=int,  default=40)
    parser.add_argument("--horizon",      type=int,  default=5)
    parser.add_argument("--n-samples",    type=int,  default=1500)
    parser.add_argument("--perplexity",   type=int,  default=30)
    return parser.parse_args()


if __name__ == "__main__":
    args = parse_args()
    plot_tsne(
        outputs_base=args.outputs_base,
        version=args.version,
        csv_file=args.csv,
        seq_len=args.seq_len,
        horizon=args.horizon,
        n_samples=args.n_samples,
        perplexity=args.perplexity,
    )


# Live Inference

In [ ]:
import argparse
import pickle
import time
from pathlib import Path
from typing import Optional, Tuple

import ccxt
import numpy as np
import pandas as pd
import torch
from torch import nn
from datetime import datetime

from dataloader import load_data
from transformer import TimeSeriesTransformer


def find_latest_version(base: str = "outputs") -> Path:
    """Return the outputs/vN folder with the highest N."""
    base_path = Path(base)
    dirs = [d for d in base_path.iterdir() if d.is_dir() and d.name.startswith("v")]
    if not dirs:
        raise FileNotFoundError(f"No versioned run folders found in '{base}'.")
    versions = []
    for d in dirs:
        try:
            versions.append((int(d.name[1:]), d))
        except ValueError:
            pass
    versions.sort(key=lambda x: x[0])
    return versions[-1][1]


def get_next_data_dir(base: str = "data") -> Path:
    """Create data/data_1, data/data_2, ... and return the new folder."""
    base_path = Path(base)
    base_path.mkdir(exist_ok=True)
    existing = [d for d in base_path.iterdir() if d.is_dir() and d.name.startswith("data_")]
    versions = []
    for d in existing:
        try:
            versions.append(int(d.name.split("_")[1]))
        except (ValueError, IndexError):
            pass
    next_n = max(versions, default=0) + 1
    out_dir = base_path / f"data_{next_n}"
    out_dir.mkdir()
    return out_dir


def load_model_and_scaler(
    version_dir: Path,
    device: torch.device,
) -> Tuple[nn.Module, object]:
    """
    Load model weights and scaler from version_dir.
    input_dim is inferred from the scaler's n_features_in_.
    Reads config.json for encoding and attn_diagonal_bias if present.
    """
    import json

    ckpt_path = version_dir / "transformer_best.pt"
    scaler_path = version_dir / "scaler.pkl"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
    if not scaler_path.exists():
        raise FileNotFoundError(f"Scaler not found: {scaler_path}")

    with open(scaler_path, "rb") as f:
        scaler = pickle.load(f)
    input_dim = scaler.n_features_in_

    config = {}
    config_path = version_dir / "config.json"
    if config_path.exists():
        with open(config_path) as f:
            config = json.load(f)

    model = TimeSeriesTransformer(
        input_dim=input_dim,
        pos_encoding=config.get("encoding", "sinusoidal"),
        attn_diagonal_bias=config.get("attn_diagonal_bias", 0.0),
    ).to(device)
    state_dict = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model, scaler


def predict_latest_direction(
    model: nn.Module,
    scaler: object,
    live_csv: str,
    seq_len: int,
    up_threshold: float = 0.6,
    down_threshold: float = 0.4,
) -> Tuple[float, int, str]:
    """
    Run inference on the most recent seq_len samples from live_csv (after
    applying the same clean.R-style processing and scaling).

    Returns (prob_up, class_label, action), where:
      - prob_up: P(price in ~30s > now)
      - class_label: 1 for up, 0 for non-up
      - action: "LONG", "SHORT", or "FLAT" (no trade)
    """
    device = next(model.parameters()).device
    df, X, y, feature_cols = load_data(live_csv)
    if len(X) < seq_len:
        raise ValueError(f"Not enough rows after cleaning to form a window of length {seq_len}.")

    X_scaled = scaler.transform(X)
    X_window = X_scaled[-seq_len:]
    X_window_t = torch.from_numpy(X_window).float().unsqueeze(0).to(device)  # (1, T, F)

    with torch.no_grad():
        logits = model(X_window_t)
        prob_up = float(torch.sigmoid(logits).item())

    label = int(prob_up >= 0.5)

    if prob_up >= up_threshold:
        action = "LONG"
    elif prob_up <= down_threshold:
        action = "SHORT"
    else:
        action = "FLAT"

    return prob_up, label, action


def main() -> None:
    parser = argparse.ArgumentParser(description="Live-ish continuous inference for Transformer model")
    parser.add_argument(
        "--version-dir",
        type=str,
        default=None,
        help="Path to outputs/vN directory (default: auto-detect latest outputs/vN)",
    )
    parser.add_argument(
        "--live-csv",
        type=str,
        default=None,
        help="CSV path (default: data/data_N/live_orderflow.csv, new folder each run).",
    )
    parser.add_argument(
        "--data-base",
        type=str,
        default="data",
        help="Base folder for data/data_N when creating new run directories.",
    )
    parser.add_argument(
        "--seq-len",
        type=int,
        default=40,
        help="Sequence length used during training.",
    )
    parser.add_argument(
        "--up-threshold",
        type=float,
        default=0.6,
        help="Minimum prob(up) to go LONG.",
    )
    parser.add_argument(
        "--down-threshold",
        type=float,
        default=0.4,
        help="Maximum prob(up) to go SHORT.",
    )
    parser.add_argument(
        "--poll-interval",
        type=float,
        default=5.0,
        help="Seconds between checks for new data / trades.",
    )
    parser.add_argument(
        "--hold-seconds",
        type=float,
        default=30.0,
        help="Seconds to hold a position before closing (matches model prediction horizon).",
    )
    parser.add_argument(
        "--symbol",
        type=str,
        default="BTC/USDT",
        help="Trading pair symbol for live data (ccxt format).",
    )
    parser.add_argument(
        "--outputs-base",
        type=str,
        default="outputs",
        help="Base folder for outputs/vN when auto-detecting latest model.",
    )
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if args.version_dir is not None:
        version_dir = Path(args.version_dir)
    else:
        version_dir = find_latest_version(args.outputs_base)
    print(f"Using model from: {version_dir}")

    model, scaler = load_model_and_scaler(version_dir, device)

    # Create fresh data folder for this run (data/data_1, data/data_2, ...)
    if args.live_csv is not None:
        live_path = Path(args.live_csv)
        live_path.parent.mkdir(parents=True, exist_ok=True)
    else:
        data_dir = get_next_data_dir(args.data_base)
        live_path = data_dir / "live_orderflow.csv"
    print(f"Live CSV: {live_path}")

    # Live data exchange (BinanceUS, same as training data source)
    exchange = ccxt.binanceus()

    print("Starting continuous inference & trading loop. Press Ctrl+C to stop.\n")

    last_n_rows = 0
    position = "FLAT"  # "FLAT", "LONG", or "SHORT"
    entry_price: float | None = None
    entry_time: datetime | None = None
    total_pnl = 0.0
    trade_count = 0

    try:
        while True:
            # 1) Fetch a fresh live order book snapshot
            try:
                ob = exchange.fetch_order_book(args.symbol)
            except Exception as e:
                print(f"[error] fetch_order_book failed: {e}")
                time.sleep(args.poll_interval)
                continue

            # Compute the same raw features as the teacher CSV header
            bids = ob["bids"]
            asks = ob["asks"]
            if not bids or not asks:
                print("[warn] Empty order book, skipping this tick.")
                time.sleep(args.poll_interval)
                continue

            best_bid, bid_vol_1 = bids[0]
            best_ask, ask_vol_1 = asks[0]
            mid_price = (best_bid + best_ask) / 2.0
            spread = best_ask - best_bid
            fractional_price = mid_price - int(mid_price)
            micro_price = (best_bid * ask_vol_1 + best_ask * bid_vol_1) / (bid_vol_1 + ask_vol_1)

            # Depth-10 OBI metrics
            top_bids_10 = bids[:10]
            top_asks_10 = asks[:10]
            bid_vol_10 = sum(b[1] for b in top_bids_10)
            ask_vol_10 = sum(a[1] for a in top_asks_10)
            if (bid_vol_10 + ask_vol_10) == 0:
                obi_10 = 0.0
            else:
                obi_10 = (bid_vol_10 - ask_vol_10) / (bid_vol_10 + ask_vol_10)
            if (bid_vol_1 + ask_vol_1) == 0:
                obi_1 = 0.0
            else:
                obi_1 = (bid_vol_1 - ask_vol_1) / (bid_vol_1 + ask_vol_1)
            obi_diff = obi_10 - obi_1

            # Append a new row to the live CSV for this run
            header_needed = not live_path.exists()
            row_df = pd.DataFrame(
                [
                    {
                        "Time": datetime.now().strftime("%H:%M:%S"),
                        "Mid Price": float(mid_price),
                        "Micro Price": float(micro_price),
                        "fractional price": float(fractional_price),
                        "spread": float(spread),
                        "obi_1": float(obi_1),
                        "obi_10": float(obi_10),
                        "obi_diff": float(obi_diff),
                    }
                ]
            )
            row_df.to_csv(live_path, mode="a", index=False, header=header_needed)

            # 2) Reload cleaned data from the live CSV and act when enough rows exist
            df, X, y, feature_cols = load_data(str(live_path))
            n_rows = len(X)

            if n_rows < args.seq_len:
                print(f"[loop] rows={n_rows} (<{args.seq_len}) | waiting for more data...")
            else:
                current_price = float(mid_price)

                if n_rows > last_n_rows:
                    last_n_rows = n_rows
                    prob_up, label, action = predict_latest_direction(
                        model=model,
                        scaler=scaler,
                        live_csv=str(live_path),
                        seq_len=args.seq_len,
                        up_threshold=args.up_threshold,
                        down_threshold=args.down_threshold,
                    )
                    direction = "UP" if label == 1 else "DOWN/NO-UP"

                    # Simple position management:
                    # - If flat and strong signal → open position.
                    # - If in position and strong opposite signal → close, report PnL, and flip.
                    # - Otherwise hold.

                    msg_prefix = f"[rows={n_rows}] price={current_price:.2f} prob(up)={prob_up:.4f} → class: {direction} → signal: {action}"

                    if position == "FLAT":
                        if action == "LONG":
                            position = "LONG"
                            entry_price = current_price
                            entry_time = datetime.now()
                            print(f"{msg_prefix} | OPEN LONG @ {entry_price:.2f}")
                        elif action == "SHORT":
                            position = "SHORT"
                            entry_price = current_price
                            entry_time = datetime.now()
                            print(f"{msg_prefix} | OPEN SHORT @ {entry_price:.2f}")
                        else:
                            print(f"{msg_prefix} | NO POSITION")
                    else:
                        # Already in a trade
                        assert entry_price is not None
                        assert entry_time is not None
                        held_seconds = (datetime.now() - entry_time).total_seconds()
                        close_time_expired = held_seconds >= args.hold_seconds
                        close_and_flip = False

                        if position == "LONG" and action == "SHORT":
                            close_and_flip = True
                        elif position == "SHORT" and action == "LONG":
                            close_and_flip = True

                        if close_time_expired or close_and_flip:
                            # Close existing position
                            if position == "LONG":
                                trade_pnl = current_price - entry_price
                            else:  # SHORT
                                trade_pnl = entry_price - current_price

                            trade_ret_pct = (trade_pnl / entry_price) * 100.0
                            total_pnl += trade_pnl
                            trade_count += 1

                            reason = f"after {held_seconds:.0f}s" if close_time_expired else "signal flip"
                            print(
                                f"{msg_prefix} | CLOSE {position} @ {current_price:.2f} "
                                f"(entry {entry_price:.2f}) {reason} → PnL {trade_pnl:.2f} "
                                f"({trade_ret_pct:.2f}%), total PnL {total_pnl:.2f} "
                                f"over {trade_count} trades"
                            )

                            if close_and_flip:
                                # Flip into new position
                                position = "LONG" if action == "LONG" else "SHORT"
                                entry_price = current_price
                                entry_time = datetime.now()
                                print(f"    → OPEN {position} @ {entry_price:.2f}")
                            else:
                                # Time-based close: go flat
                                position = "FLAT"
                                entry_price = None
                                entry_time = None
                        else:
                            # Hold existing position
                            print(f"{msg_prefix} | HOLD {position} from {entry_price:.2f}")

                # If no new rows, still check time-based close and print heartbeat
                if n_rows == last_n_rows and n_rows >= args.seq_len:
                    # Close position if hold time expired (even without new inference)
                    if position != "FLAT" and entry_price is not None and entry_time is not None:
                        held_seconds = (datetime.now() - entry_time).total_seconds()
                        if held_seconds >= args.hold_seconds:
                            if position == "LONG":
                                trade_pnl = current_price - entry_price
                            else:
                                trade_pnl = entry_price - current_price
                            trade_ret_pct = (trade_pnl / entry_price) * 100.0
                            total_pnl += trade_pnl
                            trade_count += 1
                            print(
                                f"[loop] CLOSE {position} @ {current_price:.2f} (entry {entry_price:.2f}) "
                                f"after {held_seconds:.0f}s → PnL {trade_pnl:.2f} ({trade_ret_pct:.2f}%), "
                                f"total PnL {total_pnl:.2f} over {trade_count} trades"
                            )
                            position = "FLAT"
                            entry_price = None
                            entry_time = None

                    status = f"[loop] rows={n_rows} price={current_price:.2f} | position={position}"
                    if entry_price is not None and position != "FLAT":
                        # Mark-to-market PnL of the open position
                        if position == "LONG":
                            mtm = current_price - entry_price
                        else:
                            mtm = entry_price - current_price
                        status += f" | unrealized PnL={mtm:.2f}"
                    print(status)

            # Sleep before next check
            time.sleep(args.poll_interval)
    except KeyboardInterrupt:
        print("\nStopped continuous inference loop.")


if __name__ == "__main__":
    main()



usage: ipykernel_launcher.py [-h] [--version-dir VERSION_DIR]
                             [--live-csv LIVE_CSV] [--data-base DATA_BASE]
                             [--seq-len SEQ_LEN] [--up-threshold UP_THRESHOLD]
                             [--down-threshold DOWN_THRESHOLD]
                             [--poll-interval POLL_INTERVAL]
                             [--hold-seconds HOLD_SECONDS] [--symbol SYMBOL]
                             [--outputs-base OUTPUTS_BASE]
ipykernel_launcher.py: error: unrecognized arguments: --f=/Users/spenser/Library/Jupyter/runtime/kernel-v32e546fff6bfa7ddbfd62db19208659543e0addea.json


SystemExit: 2

/opt/anaconda3/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
